# EDA: IEEE-CIS Fraud Detection

Mechanical descriptive EDA driven by `openbotrisk.eda`.

In [1]:
import sys, time, datetime as dt
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from openbotrisk.eda.loaders import load_ieee_meta
from openbotrisk.eda.descriptive import missingness_table, cardinality_table, label_balance
from openbotrisk.eda.report import write_report

DATA_DIR = REPO_ROOT / 'data' / 'ieee-fraud-detection'
REPORT_PATH = REPO_ROOT / 'working' / 'eda' / 'ieee-fraud-detection.md'

t0 = time.time()
meta = load_ieee_meta(DATA_DIR)
elapsed = time.time() - t0
tx = meta['transaction']
idn = meta['identity']
print(f'loaded in {elapsed:.1f}s; tx={tx.shape}, identity={idn.shape}')

loaded in 14.0s; tx=(590540, 394), identity=(144233, 41)


In [2]:
def fmt_size(b):
    for u in ['B','KB','MB','GB']:
        if b < 1024: return f'{b:.1f} {u}'
        b /= 1024
    return f'{b:.1f} TB'

file_lines = '\n'.join(f'- `{n}` — {fmt_size(s)}' for n, s in sorted(meta['file_sizes'].items()))
today = dt.date.today().isoformat()
access = (
    'Source: Kaggle competition `ieee-fraud-detection` (via `kaggle competitions download`).\n\n'
    f'Local path: `{meta["data_dir"]}`\n\n'
    'File format: CSV.\n\n'
    f'Date inspected: {today}.\n\n'
    'Files on disk:\n' + file_lines
)
print(access)

Source: Kaggle competition `ieee-fraud-detection` (via `kaggle competitions download`).

Local path: `/home/tsispace/Documents/GitHub/openbotrisk/data/ieee-fraud-detection`

File format: CSV.

Date inspected: 2026-05-23.

Files on disk:
- `sample_submission.csv` — 5.8 MB
- `test_identity.csv` — 24.6 MB
- `test_transaction.csv` — 584.8 MB
- `train_identity.csv` — 25.3 MB
- `train_transaction.csv` — 651.7 MB


In [3]:
# TransactionDT is seconds offset from a reference (per Kaggle: arbitrary start datetime).
# We treat it as relative seconds.
dt_min, dt_max = int(tx['TransactionDT'].min()), int(tx['TransactionDT'].max())
span_days = (dt_max - dt_min) / 86400
join_overlap = idn['TransactionID'].isin(tx['TransactionID']).sum()
structure = (
    f'- `train_transaction.csv`: {len(tx):,} rows x {tx.shape[1]} cols. One row = one transaction.\n'
    f'- `train_identity.csv`: {len(idn):,} rows x {idn.shape[1]} cols. Identity / device features attached to a subset of transactions.\n'
    f'- Join key: `TransactionID` (left join transaction <- identity). Identity available for {join_overlap:,} / {len(tx):,} transactions ({join_overlap/len(tx):.1%}).\n'
    f'- Temporal coverage: `TransactionDT` ranges from {dt_min:,} to {dt_max:,} seconds (~{span_days:.1f} days) from an unspecified reference time.'
)
print(structure)

- `train_transaction.csv`: 590,540 rows x 394 cols. One row = one transaction.
- `train_identity.csv`: 144,233 rows x 41 cols. Identity / device features attached to a subset of transactions.
- Join key: `TransactionID` (left join transaction <- identity). Identity available for 144,233 / 590,540 transactions (24.4%).
- Temporal coverage: `TransactionDT` ranges from 86,400 to 15,811,131 seconds (~182.0 days) from an unspecified reference time.


In [4]:
def schema_block(df, name, descs):
    rows = [f'### `{name}` ({df.shape[1]} columns)', '', '| column | dtype | example | description |', '|---|---|---|---|']
    samp = df.iloc[0].to_dict() if len(df) else {}
    for c in df.columns:
        v = samp.get(c, '')
        if pd.isna(v): v = ''
        v = str(v)[:30]
        d = descs.get(c)
        if d is None:
            if c.startswith('V'): d = 'Vesta-engineered numeric feature (anonymised)'
            elif c.startswith('C'): d = 'count-type feature (anonymised)'
            elif c.startswith('D'): d = 'time-delta-type feature in days (anonymised)'
            elif c.startswith('M'): d = 'match-type categorical (T/F) (anonymised)'
            elif c.startswith('id_'): d = 'identity feature (anonymised)'
            elif c.startswith('card'): d = 'payment card attribute (anonymised)'
            elif c.startswith('addr'): d = 'address attribute (anonymised)'
            elif c.startswith('dist'): d = 'distance feature (anonymised)'
            else: d = '(anonymised / undocumented)'
        rows.append(f'| `{c}` | {df[c].dtype} | `{v}` | {d} |')
    return '\n'.join(rows)

tx_descs = {
    'TransactionID': 'Unique transaction id',
    'isFraud': 'Target: 1 = fraudulent, 0 = legitimate',
    'TransactionDT': 'Time-delta from reference, in seconds',
    'TransactionAmt': 'Transaction amount (USD)',
    'ProductCD': 'Product code (categorical)',
    'P_emaildomain': 'Purchaser email domain',
    'R_emaildomain': 'Recipient email domain',
}
id_descs = {
    'TransactionID': 'Join key to train_transaction',
    'DeviceType': 'Device class (desktop / mobile)',
    'DeviceInfo': 'Device info string (browser/UA-derived)',
}
# Limit verbose V-column listing — show V1..V20 + summary line, otherwise schema explodes.
tx_show = tx[[c for c in tx.columns if not c.startswith('V') or c in [f'V{i}' for i in range(1, 21)]]]
schema_md = (
    schema_block(tx_show, 'train_transaction (showing non-V columns + V1..V20 of 339 V cols)', tx_descs)
    + f'\n\nFull V-column range: V1..V339 ({sum(c.startswith("V") for c in tx.columns)} columns).'
    + '\n\n' + schema_block(idn, 'train_identity', id_descs)
    + '\n\nMost columns (V*, C*, D*, M*, id_*, card*, addr*, dist*) are anonymised by Vesta with no public mapping.'
)
print(schema_md[:600])

### `train_transaction (showing non-V columns + V1..V20 of 339 V cols)` (75 columns)

| column | dtype | example | description |
|---|---|---|---|
| `TransactionID` | int64 | `2987000` | Unique transaction id |
| `isFraud` | int64 | `0` | Target: 1 = fraudulent, 0 = legitimate |
| `TransactionDT` | int64 | `86400` | Time-delta from reference, in seconds |
| `TransactionAmt` | float64 | `68.5` | Transaction amount (USD) |
| `ProductCD` | object | `W` | Product code (categorical) |
| `card1` | int64 | `13926` | payment card attribute (anonymised) |
| `card2` | float64 | `` | payment card attribu


In [5]:
lbl = label_balance(tx, 'isFraud').sort_values('value')
lines = ['| isFraud | count | rate |', '|---|---|---|']
for _, r in lbl.iterrows():
    lines.append(f'| {int(r["value"])} | {int(r["count"]):,} | {r["rate"]:.5f} |')
label_md = (
    'Label column: `isFraud` in `train_transaction.csv` (1 = fraudulent).\n\n'
    + '\n'.join(lines) + '\n\n'
    'Per Kaggle: a fraud label is propagated to all transactions in a card/account/email/etc. chain after the first reported fraud; '
    'so the label is partly a chain-propagated proxy rather than a per-transaction confirmation.'
)
print(label_md)

Label column: `isFraud` in `train_transaction.csv` (1 = fraudulent).

| isFraud | count | rate |
|---|---|---|
| 0 | 569,877 | 0.96501 |
| 1 | 20,663 | 0.03499 |

Per Kaggle: a fraud label is propagated to all transactions in a card/account/email/etc. chain after the first reported fraud; so the label is partly a chain-propagated proxy rather than a per-transaction confirmation.


In [6]:
ident_cols = ['TransactionID', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
              'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain']
ident_cols_id = ['DeviceType', 'DeviceInfo', 'id_30', 'id_31', 'id_33']
ct_tx = cardinality_table(tx, [c for c in ident_cols if c in tx.columns])
ct_id = cardinality_table(idn, [c for c in ident_cols_id if c in idn.columns])
ct = pd.concat([ct_tx, ct_id], ignore_index=True)
lines = ['| column | n_unique | null_rate | role |', '|---|---|---|---|']
roles = {
    'TransactionID': 'transaction primary key',
    'card1': 'card hash (weak account identifier)',
    'card2': 'card attribute',
    'card3': 'card attribute',
    'card4': 'card network (visa/mc/etc.)',
    'card5': 'card attribute',
    'card6': 'card type (debit/credit)',
    'addr1': 'billing address region',
    'addr2': 'billing country',
    'P_emaildomain': 'purchaser email domain (weak actor signal)',
    'R_emaildomain': 'recipient email domain',
    'DeviceType': 'device class',
    'DeviceInfo': 'device fingerprint string',
    'id_30': 'OS string',
    'id_31': 'browser string',
    'id_33': 'screen resolution',
}
for _, r in ct.iterrows():
    lines.append(f'| `{r["column"]}` | {int(r["n_unique"]):,} | {r["null_rate"]:.4f} | {roles.get(r["column"], "")} |')
ident_md = (
    'No explicit user ID. Weak actor identifiers come from card hashes, email domains, '
    'and (when joined) device/browser strings.\n\n'
    + '\n'.join(lines)
)
print(ident_md)

No explicit user ID. Weak actor identifiers come from card hashes, email domains, and (when joined) device/browser strings.

| column | n_unique | null_rate | role |
|---|---|---|---|
| `TransactionID` | 590,540 | 0.0000 | transaction primary key |
| `card1` | 13,553 | 0.0000 | card hash (weak account identifier) |
| `card2` | 500 | 0.0151 | card attribute |
| `card3` | 114 | 0.0027 | card attribute |
| `card4` | 4 | 0.0027 | card network (visa/mc/etc.) |
| `card5` | 119 | 0.0072 | card attribute |
| `card6` | 4 | 0.0027 | card type (debit/credit) |
| `addr1` | 332 | 0.1113 | billing address region |
| `addr2` | 74 | 0.1113 | billing country |
| `P_emaildomain` | 59 | 0.1599 | purchaser email domain (weak actor signal) |
| `R_emaildomain` | 60 | 0.7675 | recipient email domain |
| `DeviceType` | 2 | 0.0237 | device class |
| `DeviceInfo` | 1,786 | 0.1773 | device fingerprint string |
| `id_30` | 75 | 0.4622 | OS string |
| `id_31` | 130 | 0.0274 | browser string |
| `id_33` | 260 | 0.4

In [7]:
temporal_md = (
    f'- `TransactionDT`: integer seconds from a reference time (reference not disclosed).\n'
    f'- Range: {dt_min:,} to {dt_max:,} seconds ({span_days:.1f} days of activity).\n'
    '- Granularity: 1 second. No wall-clock timestamps; no timezone.\n'
    '- D1..D15 are documented as time-delta features in days relative to prior events on the same card / account.\n'
    '- Diurnal patterns are inferable modulo the unknown reference time; not plotted here.'
)
print(temporal_md)

- `TransactionDT`: integer seconds from a reference time (reference not disclosed).
- Range: 86,400 to 15,811,131 seconds (182.0 days of activity).
- Granularity: 1 second. No wall-clock timestamps; no timezone.
- D1..D15 are documented as time-delta features in days relative to prior events on the same card / account.
- Diurnal patterns are inferable modulo the unknown reference time; not plotted here.


In [8]:
mt_tx = missingness_table(tx)
mt_id = missingness_table(idn)
hi_tx = mt_tx[mt_tx['null_rate'] > 0.2]
hi_id = mt_id[mt_id['null_rate'] > 0.2]
missing_md = (
    f'- `train_transaction`: {tx.isna().sum().sum():,} null cells overall '
    f'({tx.isna().sum().sum() / tx.size:.2%} of cells). '
    f'{len(hi_tx)} of {tx.shape[1]} columns have >20% missingness.\n'
    f'- `train_identity`: {idn.isna().sum().sum():,} null cells overall '
    f'({idn.isna().sum().sum() / idn.size:.2%}). '
    f'{len(hi_id)} of {idn.shape[1]} columns have >20% missingness.\n'
    f'- Identity rows themselves are missing for ~{1 - join_overlap/len(tx):.0%} of transactions '
    '(no identity file row for that TransactionID).\n\n'
    f'Top 15 most-null columns in `train_transaction`:\n\n'
    + '| column | null_rate |\n|---|---|\n'
    + '\n'.join(f'| `{r["column"]}` | {r["null_rate"]:.4f} |' for _, r in mt_tx.head(15).iterrows())
    + '\n\nTop 15 most-null columns in `train_identity`:\n\n'
    + '| column | null_rate |\n|---|---|\n'
    + '\n'.join(f'| `{r["column"]}` | {r["null_rate"]:.4f} |' for _, r in mt_id.head(15).iterrows())
)
print(missing_md[:600])

- `train_transaction`: 95,566,686 null cells overall (41.07% of cells). 212 of 394 columns have >20% missingness.
- `train_identity`: 2,104,107 null cells overall (35.58%). 19 of 41 columns have >20% missingness.
- Identity rows themselves are missing for ~76% of transactions (no identity file row for that TransactionID).

Top 15 most-null columns in `train_transaction`:

| column | null_rate |
|---|---|
| `dist2` | 0.9363 |
| `D7` | 0.9341 |
| `D13` | 0.8951 |
| `D14` | 0.8947 |
| `D12` | 0.8904 |
| `D6` | 0.8761 |
| `D9` | 0.8731 |
| `D8` | 0.8731 |
| `V153` | 0.8612 |
| `V149` | 0.8612 |
| 


In [9]:
quirks_md = (
    f'- Two-file structure (transaction + identity) joined on `TransactionID`; only {join_overlap/len(tx):.0%} of transactions have identity rows.\n'
    '- V*/C*/D*/M*/id_* columns are anonymised; no public mapping. Treat them as opaque features.\n'
    '- `TransactionDT` is seconds from an unstated reference time; cannot anchor to calendar dates.\n'
    '- Labels are chain-propagated (per Kaggle): once a card/account is flagged, related transactions inherit the positive label, which inflates positives and complicates per-row interpretation.\n'
    '- High missingness is concentrated in long blocks of V/D/id_ columns — typical of features only defined for certain product codes or device classes.'
)
print(quirks_md)

- Two-file structure (transaction + identity) joined on `TransactionID`; only 24% of transactions have identity rows.
- V*/C*/D*/M*/id_* columns are anonymised; no public mapping. Treat them as opaque features.
- `TransactionDT` is seconds from an unstated reference time; cannot anchor to calendar dates.
- Labels are chain-propagated (per Kaggle): once a card/account is flagged, related transactions inherit the positive label, which inflates positives and complicates per-row interpretation.
- High missingness is concentrated in long blocks of V/D/id_ columns — typical of features only defined for certain product codes or device classes.


In [10]:
repro_md = (
    'Generated by `notebooks/eda/ieee-fraud-detection.ipynb` which calls '
    '`openbotrisk.eda.loaders.load_ieee_meta` (pandas full-read).\n\n'
    'Run with:\n\n'
    '```bash\n'
    'jupyter nbconvert --to notebook --execute --inplace \\\n'
    '  notebooks/eda/ieee-fraud-detection.ipynb \\\n'
    '  --ExecutePreprocessor.timeout=600\n'
    '```\n\n'
    f'Loader runtime on this machine: {elapsed:.1f}s. '
    'Both train CSVs fit in memory; no chunking needed.'
)
print(repro_md)

Generated by `notebooks/eda/ieee-fraud-detection.ipynb` which calls `openbotrisk.eda.loaders.load_ieee_meta` (pandas full-read).

Run with:

```bash
jupyter nbconvert --to notebook --execute --inplace \
  notebooks/eda/ieee-fraud-detection.ipynb \
  --ExecutePreprocessor.timeout=600
```

Loader runtime on this machine: 14.0s. Both train CSVs fit in memory; no chunking needed.


In [11]:
stats = {
    'Access': access,
    'Structure': structure,
    'Schema': schema_md,
    'Label': label_md,
    'Identifier inventory': ident_md,
    'Temporal structure': temporal_md,
    'Missing data': missing_md,
    'Quirks and observations': quirks_md,
    'Reproduction': repro_md,
}
out = write_report(stats, 'IEEE-CIS Fraud Detection', REPORT_PATH)
print('wrote', out)

wrote /home/tsispace/Documents/GitHub/openbotrisk/working/eda/ieee-fraud-detection.md
